# Tokenization Basics in TensorFlow

This notebook gives a practical introduction to tokenization.

We will cover:
- What tokenization means in simple words
- Real-world examples and use-cases
- Different tokenization types in `tensorflow-text`
- Runnable code examples for each tokenizer

## 1) Concept in Simple Words

Tokenization means breaking text into smaller parts called tokens.

Think of it like cutting a pizza:
- The full pizza is a sentence
- Each slice is a token

Depending on your model and task, a token can be:
- A whole sentence
- A word
- A subword piece
- A single character

### Real-Time Examples
- Chat apps: split messages into words to detect intent
- Search engines: split query text to match documents faster
- Translation apps: split sentence pieces so unknown words can still be handled
- Spam filters: split email text to find suspicious patterns

### Why This Matters
Neural networks work with numbers, not raw text.
Tokenization is the first step that turns human language into machine-readable units.

In [1]:
import tensorflow as tf
import tensorflow_text as tf_text

print('TensorFlow version:', tf.__version__)

sample_texts = tf.constant([
    'I love TensorFlow',
    'Tokenization helps NLP models'
])

sample_texts

TensorFlow version: 2.22.0-dev0+selfbuilt


<tf.Tensor: shape=(2,), dtype=string, numpy=
array([b'I love TensorFlow', b'Tokenization helps NLP models'],
      dtype=object)>

In [ ]:
import importlib.util
import os


def load_runtime_env() -> None:
    candidate_paths = [
        os.path.join(os.getcwd(), "configs", "runtime.env"),
        os.path.join(os.getcwd(), "configs", "runtime.env.example"),
        os.path.join(os.getcwd(), "..", "configs", "runtime.env"),
        os.path.join(os.getcwd(), "..", "configs", "runtime.env.example"),
    ]
    for path in candidate_paths:
        if not os.path.exists(path):
            continue
        with open(path, "r", encoding="utf-8") as handle:
            for raw_line in handle:
                line = raw_line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                key, value = line.split("=", 1)
                key = key.strip()
                value = value.strip().strip("\"'")
                if key and key not in os.environ:
                    os.environ[key] = value
        break


def parse_use_gpu_flag(raw_value: str) -> bool:
    normalized = raw_value.strip().lower()
    return normalized not in {"0", "false", "no", "off"}


load_runtime_env()
USE_GPU = parse_use_gpu_flag(os.getenv("USE_GPU", "1"))
NO_CUDA = not USE_GPU

_torch_cuda = False
_tf_gpu = False
if importlib.util.find_spec("torch") is not None:
    import torch

    _torch_cuda = torch.cuda.is_available()
if importlib.util.find_spec("tensorflow") is not None:
    import tensorflow as tf

    _tf_gpu = bool(tf.config.list_physical_devices("GPU"))

RUNTIME_DEVICE = "cuda" if USE_GPU and (_torch_cuda or _tf_gpu) else "cpu"
print(f"USE_GPU={int(USE_GPU)} | runtime_device={RUNTIME_DEVICE}")

## 2) Types of Tokenization in `tensorflow-text`

We will try these tokenizer types:
1. Whitespace tokenization using `tf_text.WhitespaceTokenizer`
2. Unicode script tokenization using `tf_text.UnicodeScriptTokenizer`
3. Subword tokenization (WordPiece) using `tf_text.WordpieceTokenizer`

We will also discuss where SentencePiece and BERT tokenizers fit in.

### A) Whitespace Tokenizer

This is the most basic style.
It splits text by spaces.

Use-case:
- Fast baseline
- Clean English-like text where spacing is reliable

In [2]:
whitespace_tokenizer = tf_text.WhitespaceTokenizer()
whitespace_tokens = whitespace_tokenizer.tokenize(sample_texts)

print('Whitespace tokens:')
print(whitespace_tokens.to_list())

Whitespace tokens:
[[b'I', b'love', b'TensorFlow'], [b'Tokenization', b'helps', b'NLP', b'models']]


### B) Unicode Script Tokenizer

This tokenizer is better for multilingual text.
It splits tokens based on Unicode script boundaries and punctuation.

Use-case:
- Mixed-language text
- Inputs containing punctuation-heavy content

In [3]:
multilingual_text = tf.constant([
    'Hello, world! 123',
    'Namaste दुनिया, TensorFlow!'
])

unicode_script_tokenizer = tf_text.UnicodeScriptTokenizer()
unicode_tokens = unicode_script_tokenizer.tokenize(multilingual_text)

print('Unicode script tokens:')
print(unicode_tokens.to_list())

Unicode script tokens:
[[b'Hello', b',', b'world', b'!123'], [b'Namaste', b'\xe0\xa4\xa6\xe0\xa5\x81\xe0\xa4\xa8\xe0\xa4\xbf\xe0\xa4\xaf\xe0\xa4\xbe', b',', b'TensorFlow', b'!']]


### C) WordPiece (Subword) Tokenizer

WordPiece breaks rare words into smaller known parts.
Example: `playing` can become `play` + `##ing`.

Why this helps:
- Handles unknown words better than word-level tokenization
- Commonly used in BERT-style NLP pipelines

In [4]:
vocab = [
    '[PAD]', '[UNK]',
    'i', 'love', 'machine', 'learn', 'play',
    '##ing', 'ten', '##sor', '##flow'
]

initializer = tf.lookup.KeyValueTensorInitializer(
    keys=tf.constant(vocab),
    values=tf.range(len(vocab), dtype=tf.int64)
)

vocab_table = tf.lookup.StaticVocabularyTable(
    initializer=initializer,
    num_oov_buckets=1
)

wordpiece_tokenizer = tf_text.WordpieceTokenizer(
    vocab_lookup_table=vocab_table,
    token_out_type=tf.string,
    unknown_token='[UNK]'
)

# WordPiece expects pre-tokenized words (ragged input)
pretokenized_words = tf.ragged.constant([
    ['i', 'love', 'tensorflow'],
    ['machine', 'learning', 'playing']
])

wordpiece_tokens = wordpiece_tokenizer.tokenize(pretokenized_words)

print('WordPiece tokens:')
print(wordpiece_tokens.to_list())

WordPiece tokens:
[[[b'i'], [b'love'], [b'ten', b'##sor', b'##flow']], [[b'machine'], [b'learn', b'##ing'], [b'play', b'##ing']]]


## 6) Each Tokenizer: Pros, Cons, and When to Use

| Tokenizer | Pros | Cons | Best Time to Use |
|---|---|---|---|
| WhitespaceTokenizer | Very fast, easiest to understand, great for teaching/baselines | Splits only on spaces, weak for punctuation-heavy text and languages without space separators | Quick baseline, clean English-like logs/search text |
| UnicodeScriptTokenizer | Better multilingual splitting, handles punctuation/script boundaries more robustly | Still not true subword tokenization, may not match pretrained model tokenization exactly | Mixed-language user text, social data, multilingual preprocessing |
| WordpieceTokenizer | Strong for unknown words via subword units, standard in many NLP systems | Needs controlled vocabulary, setup is more complex than word-level tokenizers | Production NLP, classification, retrieval, BERT-style pipelines |
| SentencepieceTokenizer | Language-agnostic, robust where whitespace is unreliable, works well for multilingual corpora | Requires trained SentencePiece model artifact and lifecycle management | Large multilingual datasets, MT/LLM style data pipelines |
| BertTokenizer | Compatible with BERT preprocessing expectations, includes basic + WordPiece behavior | Requires BERT-compatible vocab/config; often coupled to a model family | Fine-tuning or serving BERT and BERT-derived models |

## 7) Quick Use-Case Mapping

| Problem | Recommended Tokenizer |
|---|---|
| Quick prototype for text classification | WhitespaceTokenizer -> then upgrade to WordPiece |
| Multilingual moderation and noisy user text | UnicodeScriptTokenizer or SentencePiece |
| BERT fine-tuning (sentiment, NER, intent) | BertTokenizer |
| Domain-specific NLP with OOV terms | WordPiece with regularly refreshed vocabulary |
| Language-agnostic large-scale pipeline | SentencePiece |

### D) SentencePiece Tokenizer

SentencePiece is a language-independent subword tokenizer.
It uses a trained `.model` file and is popular for multilingual pipelines.

Use-case:
- Translation systems
- Large multilingual language models
- Pipelines where whitespace rules are unreliable

Below is a usage example with a pre-trained SentencePiece model file path.

In [5]:
# Requires a valid sentencepiece model file (.model) generated offline.
# Example path only: update before running.
sp_model_path = 'tokenizers/sentencepiece.model'

try:
    sp_model = tf.io.gfile.GFile(sp_model_path, 'rb').read()
    sentencepiece_tokenizer = tf_text.SentencepieceTokenizer(model=sp_model)

    sp_input = tf.constant([
        'Tokenization with SentencePiece',
        'It works well for multilingual text'
    ])

    sp_tokens = sentencepiece_tokenizer.tokenize(sp_input)
    print('SentencePiece token ids:')
    print(sp_tokens.to_list())
except Exception as e:
    print('SentencePiece example needs a valid .model file at:', sp_model_path)
    print('Error:', e)

SentencePiece example needs a valid .model file at: tokenizers/sentencepiece.model
Error: tokenizers/sentencepiece.model; No such file or directory


### E) BERT Tokenizer

`BertTokenizer` is useful when your model expects BERT-style tokenization.
It usually lowercases text (if configured), splits punctuation, and applies WordPiece.

Use-case:
- BERT fine-tuning for sentiment, classification, NER
- Any pipeline that needs BERT-compatible tokens

In [6]:
bert_vocab = [
    '[PAD]', '[UNK]', '[CLS]', '[SEP]', '[MASK]',
    'i', 'love', 'tensor', '##flow', 'nlp', '.', '##s'
]

bert_initializer = tf.lookup.KeyValueTensorInitializer(
    keys=tf.constant(bert_vocab),
    values=tf.range(len(bert_vocab), dtype=tf.int64)
)

bert_vocab_table = tf.lookup.StaticVocabularyTable(
    initializer=bert_initializer,
    num_oov_buckets=1
)

bert_tokenizer = tf_text.BertTokenizer(
    vocab_lookup_table=bert_vocab_table,
    token_out_type=tf.string,
    lower_case=True
)

bert_input = tf.constant([
    'I love TensorFlow.',
    'NLPs are useful.'
])

bert_tokens = bert_tokenizer.tokenize(bert_input)
print('BERT tokens:')
print(bert_tokens.to_list())

BERT tokens:
[[[b'i'], [b'love'], [b'tensor', b'##flow'], [b'.']], [[b'nlp', b'##s'], [b'[UNK]'], [b'[UNK]'], [b'.']]]


## 5) Special Tokens and How to Use Them

Special tokens are reserved markers used by models for structure and control.

Common special tokens:
- `[PAD]`: pad shorter sequences to equal length
- `[UNK]`: unknown token fallback
- `[CLS]`: classification start token (BERT)
- `[SEP]`: sentence separator (BERT)
- `[MASK]`: masked position token (BERT pretraining)

### Special Token Mapping by Tokenizer

| Tokenizer | Typical Special Tokens | How to Use |
|---|---|---|
| WhitespaceTokenizer | Usually none by default | Add special tokens manually after tokenization |
| UnicodeScriptTokenizer | Usually none by default | Add special tokens manually after tokenization |
| WordpieceTokenizer | `[UNK]`, optionally `[PAD]`, `[CLS]`, `[SEP]`, `[MASK]` | Include tokens in vocab and add control tokens in input pipeline |
| SentencepieceTokenizer | `<unk>`, `<pad>`, `<s>`, `</s>` (depends on model config) | Configure during SentencePiece training; tokenizer emits ids accordingly |
| BertTokenizer | `[PAD]`, `[UNK]`, `[CLS]`, `[SEP]`, `[MASK]` | Keep these in BERT vocab and compose model input with them |

In [7]:
# Example: adding BERT control tokens to a tokenized sequence
raw_words = tf.ragged.constant([['i', 'love', 'tensor', '##flow']])

cls_sep_wrapped = tf.concat(
    [
        tf.ragged.constant([['[CLS]']]),
        raw_words,
        tf.ragged.constant([['[SEP]']])
    ],
    axis=1
)

print('With [CLS]/[SEP]:')
print(cls_sep_wrapped.to_list())

# Optional fixed-length padding with [PAD]
target_len = 8
padded = cls_sep_wrapped.to_tensor(default_value='[PAD]', shape=[1, target_len])
print('Padded sequence:')
print(padded.numpy())

With [CLS]/[SEP]:
[[b'[CLS]', b'i', b'love', b'tensor', b'##flow', b'[SEP]']]
Padded sequence:
[[b'[CLS]' b'i' b'love' b'tensor' b'##flow' b'[SEP]' b'[PAD]' b'[PAD]']]


## 5) Static vs Dynamic Tokenizers

### Static Tokenizer
A static tokenizer uses a fixed vocabulary/model that does not change while training or serving.

Examples:
- WordpieceTokenizer with fixed vocab table
- SentencepieceTokenizer with fixed model file
- BertTokenizer with fixed BERT vocab

Pros:
- Reproducible and stable behavior
- Easy to version and deploy

Cons:
- New domain terms may become unknown or split poorly

### Dynamic Tokenizer
A dynamic tokenizer updates vocabulary from new data periodically.
In practical TensorFlow pipelines, this is usually done offline:
1. Collect fresh text data
2. Rebuild or refresh tokenizer vocabulary/model
3. Version and redeploy tokenizer artifacts

Pros:
- Adapts to changing language, product names, and trends

Cons:
- Needs artifact governance, versioning, and retraining discipline

In [8]:
# Static vocabulary (fixed list)
static_vocab = ['[PAD]', '[UNK]', 'data', 'science', 'model']
static_table = tf.lookup.StaticVocabularyTable(
    tf.lookup.KeyValueTensorInitializer(
        keys=tf.constant(static_vocab),
        values=tf.range(len(static_vocab), dtype=tf.int64)
    ),
    num_oov_buckets=1
)

static_wp = tf_text.WordpieceTokenizer(
    vocab_lookup_table=static_table,
    token_out_type=tf.string,
    unknown_token='[UNK]'
)

print('Static tokenizer output:')
print(static_wp.tokenize(tf.ragged.constant([['data', 'scientist']])).to_list())

# Dynamic vocabulary pattern (recomputed from new corpus offline)
new_corpus = tf.constant([
    'data science model',
    'data engineering pipeline',
    'model deployment model'
])

# Build a simple frequency vocab dynamically from corpus tokens
dynamic_words = tf_text.WhitespaceTokenizer().tokenize(new_corpus).flat_values
unique_words, _, counts = tf.unique_with_counts(dynamic_words)

# Keep top-k frequent words (toy demonstration)
k = tf.minimum(6, tf.size(unique_words))
top_indices = tf.argsort(counts, direction='DESCENDING')[:k]
top_words = tf.gather(unique_words, top_indices)

dynamic_vocab = tf.concat(
    [tf.constant(['[PAD]', '[UNK]']), top_words],
    axis=0
)

dynamic_table = tf.lookup.StaticVocabularyTable(
    tf.lookup.KeyValueTensorInitializer(
        keys=dynamic_vocab,
        values=tf.range(tf.size(dynamic_vocab), dtype=tf.int64)
    ),
    num_oov_buckets=1
)

print('Dynamic vocab learned from corpus:')
print(dynamic_vocab.numpy())

Static tokenizer output:
[[[b'data'], [b'[UNK]']]]
Dynamic vocab learned from corpus:
[b'[PAD]' b'[UNK]' b'model' b'data' b'science' b'engineering' b'pipeline'
 b'deployment']


## 8) Dynamic Tokenizer Examples

Below are practical dynamic patterns:
- Example A: refresh vocabulary from new corpus snapshots
- Example B: compare old-vs-new tokenizer behavior on incoming text

These examples keep the tokenizer object static per deployment, but dynamically rebuild artifacts between versions (v1, v2, ...).

In [9]:
def build_dynamic_wordpiece_table(corpus, top_k=8):
    tokens = tf_text.WhitespaceTokenizer().tokenize(corpus).flat_values
    unique_words, _, counts = tf.unique_with_counts(tokens)
    k = tf.minimum(top_k, tf.size(unique_words))
    top_indices = tf.argsort(counts, direction='DESCENDING')[:k]
    top_words = tf.gather(unique_words, top_indices)

    vocab = tf.concat([tf.constant(['[PAD]', '[UNK]']), top_words], axis=0)
    table = tf.lookup.StaticVocabularyTable(
        tf.lookup.KeyValueTensorInitializer(
            keys=vocab,
            values=tf.range(tf.size(vocab), dtype=tf.int64)
        ),
        num_oov_buckets=1
    )
    return vocab, table

# Dynamic refresh: version v1
corpus_v1 = tf.constant([
    'great camera phone',
    'battery backup great',
    'phone camera quality'
])
vocab_v1, table_v1 = build_dynamic_wordpiece_table(corpus_v1, top_k=6)
wp_v1 = tf_text.WordpieceTokenizer(table_v1, token_out_type=tf.string, unknown_token='[UNK]')

# Dynamic refresh: version v2 (new trend words appear)
corpus_v2 = tf.constant([
    'great camera phone',
    'battery backup solid',
    'phone ai features',
    'ai camera enhancement'
])
vocab_v2, table_v2 = build_dynamic_wordpiece_table(corpus_v2, top_k=6)
wp_v2 = tf_text.WordpieceTokenizer(table_v2, token_out_type=tf.string, unknown_token='[UNK]')

incoming = tf.ragged.constant([
    ['phone', 'ai', 'camera'],
    ['battery', 'backup', 'solid']
])

print('Dynamic vocab v1:')
print(vocab_v1.numpy())
print('Dynamic vocab v2:')
print(vocab_v2.numpy())

print('Tokenization with v1 tokenizer:')
print(wp_v1.tokenize(incoming).to_list())
print('Tokenization with v2 tokenizer:')
print(wp_v2.tokenize(incoming).to_list())

Dynamic vocab v1:
[b'[PAD]' b'[UNK]' b'great' b'camera' b'phone' b'battery' b'backup'
 b'quality']
Dynamic vocab v2:
[b'[PAD]' b'[UNK]' b'camera' b'phone' b'ai' b'great' b'battery' b'backup']
Tokenization with v1 tokenizer:
[[[b'phone'], [b'[UNK]'], [b'camera']], [[b'battery'], [b'backup'], [b'[UNK]']]]
Tokenization with v2 tokenizer:
[[[b'phone'], [b'ai'], [b'camera']], [[b'battery'], [b'backup'], [b'[UNK]']]]


## 9) Special Cases: Which Tokenizer Handles What Better

| Special Case | Why It Is Tricky | Better Choice | Notes |
|---|---|---|---|
| Languages without spaces (Chinese/Japanese) | Whitespace split is unreliable | SentencePiece or WordPiece/BERT family | Subword tokenization is usually safer |
| Noisy social text (`wowww!!!`, hashtags, mixed scripts) | Punctuation and script boundaries vary | UnicodeScriptTokenizer -> then subword tokenizer | Good first split for mixed-script cleanup |
| New product terms or trending slang | High out-of-vocabulary risk | Dynamic WordPiece refresh workflow | Rebuild vocab periodically from fresh data |
| Strict reproducibility in production | Tokenizer drift can break consistency | Static tokenizer artifacts | Version vocab/model with model checkpoints |
| BERT model serving | Input format/tokenization must match pretraining | BertTokenizer | Keep vocab and casing rules identical to model family |
| Domain with frequent acronyms/codes | Word splits may lose intent | WordPiece with domain vocab | Add frequent domain tokens to vocab refresh set |

In [10]:
# Special-case demo: noisy mixed-script text
special_text = tf.constant([
    'Big SALE!!! on AI-camera 📸 #NewLaunch',
    'नया phone model2026 is awesome!!!',
    '今日はAI cameraがすごい'
])

print('WhitespaceTokenizer special-case output:')
print(tf_text.WhitespaceTokenizer().tokenize(special_text).to_list())

print('\nUnicodeScriptTokenizer special-case output:')
print(tf_text.UnicodeScriptTokenizer().tokenize(special_text).to_list())

print('\nObservation: UnicodeScriptTokenizer usually separates punctuation/script boundaries better than whitespace splitting.')

WhitespaceTokenizer special-case output:
[[b'Big', b'SALE!!!', b'on', b'AI-camera', b'\xf0\x9f\x93\xb8', b'#NewLaunch'], [b'\xe0\xa4\xa8\xe0\xa4\xaf\xe0\xa4\xbe', b'phone', b'model2026', b'is', b'awesome!!!'], [b'\xe4\xbb\x8a\xe6\x97\xa5\xe3\x81\xafAI', b'camera\xe3\x81\x8c\xe3\x81\x99\xe3\x81\x94\xe3\x81\x84']]

UnicodeScriptTokenizer special-case output:
[[b'Big', b'SALE', b'!!!', b'on', b'AI', b'-', b'camera', b'\xf0\x9f\x93\xb8#', b'NewLaunch'], [b'\xe0\xa4\xa8\xe0\xa4\xaf\xe0\xa4\xbe', b'phone', b'model', b'2026', b'is', b'awesome', b'!!!'], [b'\xe4\xbb\x8a\xe6\x97\xa5', b'\xe3\x81\xaf', b'AI', b'camera', b'\xe3\x81\x8c\xe3\x81\x99\xe3\x81\x94\xe3\x81\x84']]

Observation: UnicodeScriptTokenizer usually separates punctuation/script boundaries better than whitespace splitting.


## 10) Are Special Tokens Implemented Differently in WordPiece vs SentencePiece?

Yes. The implementation style is different.

### WordpieceTokenizer (vocab-driven + manual sequence formatting)
- You define special tokens directly in your vocabulary (for example `[PAD]`, `[UNK]`, `[CLS]`, `[SEP]`, `[MASK]`).
- Then you manually compose final model input by adding `[CLS]` at start and `[SEP]` at end.
- Padding is also usually done manually in the input pipeline.

### SentencepieceTokenizer (model-driven special pieces)
- Special tokens are defined inside the trained SentencePiece model (commonly `<unk>`, `<pad>`, `<s>`, `</s>`).
- You query special token ids from the tokenizer/model.
- Then you format encoded id sequences using those ids (for example adding BOS/EOS ids).

In short:
- WordPiece: special tokens are mostly vocabulary + pipeline convention.
- SentencePiece: special tokens are mostly model-defined pieces + id-level formatting.

In [11]:
# Side-by-side implementation examples

# -----------------------------
# A) WordPiece special tokens
# -----------------------------
wp_demo_vocab = [
    '[PAD]', '[UNK]', '[CLS]', '[SEP]', '[MASK]',
    'i', 'like', 'tensor', '##flow'
]

wp_demo_table = tf.lookup.StaticVocabularyTable(
    tf.lookup.KeyValueTensorInitializer(
        keys=tf.constant(wp_demo_vocab),
        values=tf.range(len(wp_demo_vocab), dtype=tf.int64)
    ),
    num_oov_buckets=1
)

wp_demo = tf_text.WordpieceTokenizer(
    vocab_lookup_table=wp_demo_table,
    token_out_type=tf.string,
    unknown_token='[UNK]'
)

wp_words = tf.ragged.constant([['i', 'like', 'tensorflow']])
# Shape is [batch, words, pieces]; flatten words->pieces to get sentence-level pieces.
wp_pieces_3d = wp_demo.tokenize(wp_words)
wp_pieces = wp_pieces_3d.merge_dims(1, 2)

wp_with_cls_sep = tf.concat(
    [tf.ragged.constant([['[CLS]']]), wp_pieces, tf.ragged.constant([['[SEP]']])],
    axis=1
)
wp_padded = wp_with_cls_sep.to_tensor(default_value='[PAD]', shape=[1, 7])

print('WordPiece pieces:')
print(wp_pieces.to_list())
print('WordPiece with [CLS]/[SEP]:')
print(wp_with_cls_sep.to_list())
print('WordPiece padded:')
print(wp_padded.numpy())

# -----------------------------------------
# B) SentencePiece special tokens (id-based)
# -----------------------------------------
# SentencePiece depends on a trained .model file that already defines special pieces.
# We query ids from that model and then compose BOS/EOS around encoded ids.

sp_model_path = 'tokenizers/sentencepiece.model'  # update to your real model path

try:
    sp_model_bytes = tf.io.gfile.GFile(sp_model_path, 'rb').read()
    sp_demo = tf_text.SentencepieceTokenizer(model=sp_model_bytes)

    bos_id = sp_demo.string_to_id(tf.constant('<s>'))
    eos_id = sp_demo.string_to_id(tf.constant('</s>'))
    pad_id = sp_demo.string_to_id(tf.constant('<pad>'))
    unk_id = sp_demo.string_to_id(tf.constant('<unk>'))

    sp_input = tf.constant(['I like TensorFlow'])
    sp_ids = sp_demo.tokenize(sp_input)

    # Add BOS/EOS at id level
    sp_dense = sp_ids.to_tensor()
    bos_col = tf.fill([tf.shape(sp_dense)[0], 1], tf.cast(bos_id, sp_dense.dtype))
    eos_col = tf.fill([tf.shape(sp_dense)[0], 1], tf.cast(eos_id, sp_dense.dtype))
    sp_with_bos_eos = tf.concat([bos_col, sp_dense, eos_col], axis=1)

    print('\nSentencePiece special ids:')
    print('bos_id:', int(bos_id.numpy()), 'eos_id:', int(eos_id.numpy()), 'pad_id:', int(pad_id.numpy()), 'unk_id:', int(unk_id.numpy()))
    print('SentencePiece token ids:')
    print(sp_ids.to_list())
    print('SentencePiece with BOS/EOS ids:')
    print(sp_with_bos_eos.numpy())
except Exception as e:
    print('\nSentencePiece example skipped. Provide a valid model at:', sp_model_path)
    print('Error:', e)

WordPiece pieces:
[[b'i', b'like', b'tensor', b'##flow']]
WordPiece with [CLS]/[SEP]:
[[b'[CLS]', b'i', b'like', b'tensor', b'##flow', b'[SEP]']]
WordPiece padded:
[[b'[CLS]' b'i' b'like' b'tensor' b'##flow' b'[SEP]' b'[PAD]']]

SentencePiece example skipped. Provide a valid model at: tokenizers/sentencepiece.model
Error: tokenizers/sentencepiece.model; No such file or directory


## 11) tf.lookup Examples (With OOV)

This section explains core `tf.lookup` patterns in simple words.

How to read these examples even without running:
- `lookup` table means: "find id/value for this token/key"
- OOV means "out of vocabulary" (token not present in table)
- We show two OOV styles:
  - single fallback value (example: always `-1`)
  - OOV buckets (example: unknown words spread across bucket ids)

`tf.lookup` APIs covered here:
- `KeyValueTensorInitializer`
- `StaticHashTable`
- `StaticVocabularyTable`
- `TextFileInitializer`
- `tf.lookup.experimental.MutableHashTable`

### Example 1: `StaticHashTable` with one OOV fallback value

What this does in simple words:
- We make a small dictionary: `apple -> 10`, `banana -> 20`, `orange -> 30`
- If a key is not found, table returns `-1`
- So unknown keys like `grape` become OOV and map to `-1`

In [12]:
keys = tf.constant(['apple', 'banana', 'orange'])
values = tf.constant([10, 20, 30], dtype=tf.int64)

kv_init = tf.lookup.KeyValueTensorInitializer(keys=keys, values=values)
fruit_table = tf.lookup.StaticHashTable(initializer=kv_init, default_value=-1)

fruit_query = tf.constant(['banana', 'grape', 'orange', 'mango'])
fruit_ids = fruit_table.lookup(fruit_query)

print('Query:', fruit_query.numpy())
print('Lookup ids:', fruit_ids.numpy())
print('Meaning: grape and mango are OOV, so they become -1')

Query: [b'banana' b'grape' b'orange' b'mango']
Lookup ids: [20 -1 30 -1]
Meaning: grape and mango are OOV, so they become -1


### Example 2: `StaticVocabularyTable` with OOV buckets

What this does in simple words:
- Known words get fixed ids from the vocabulary.
- Unknown words do not all go to one id.
- Unknown words are distributed into OOV buckets (here: 2 buckets), useful for modeling different unknowns.

In [13]:
base_vocab = tf.constant(['red', 'green', 'blue'])
base_ids = tf.constant([0, 1, 2], dtype=tf.int64)

vocab_init = tf.lookup.KeyValueTensorInitializer(keys=base_vocab, values=base_ids)
color_vocab_table = tf.lookup.StaticVocabularyTable(
    initializer=vocab_init,
    num_oov_buckets=2
)

color_query = tf.constant(['green', 'yellow', 'blue', 'purple', 'red'])
color_ids = color_vocab_table.lookup(color_query)

print('Query:', color_query.numpy())
print('Lookup ids:', color_ids.numpy())
print('Meaning: yellow/purple are OOV and map to bucket ids >= vocab size (3).')

Query: [b'green' b'yellow' b'blue' b'purple' b'red']
Lookup ids: [1 4 2 3 0]
Meaning: yellow/purple are OOV and map to bucket ids >= vocab size (3).


### Example 3: `TextFileInitializer` + `StaticHashTable`

What this does in simple words:
- We create a vocab file with one token per line.
- TensorFlow reads that file into a lookup table.
- Token id is line number (0-based here).
- Unknown tokens become OOV and return fallback `-1`.

In [14]:
import os

lookup_dir = 'tmp_lookup'
tf.io.gfile.makedirs(lookup_dir)
vocab_file = os.path.join(lookup_dir, 'vocab.txt')

# One token per line. Line number becomes id: cat->0, dog->1, bird->2
tf.io.gfile.GFile(vocab_file, 'w').write('cat\ndog\nbird\n')

file_init = tf.lookup.TextFileInitializer(
    filename=vocab_file,
    key_dtype=tf.string,
    key_index=tf.lookup.TextFileIndex.WHOLE_LINE,
    value_dtype=tf.int64,
    value_index=tf.lookup.TextFileIndex.LINE_NUMBER
)

animal_table = tf.lookup.StaticHashTable(initializer=file_init, default_value=-1)

animal_query = tf.constant(['dog', 'lion', 'bird'])
animal_ids = animal_table.lookup(animal_query)

print('Query:', animal_query.numpy())
print('Lookup ids:', animal_ids.numpy())
print('Meaning: lion is OOV, so it becomes -1')

Query: [b'dog' b'lion' b'bird']
Lookup ids: [ 1 -1  2]
Meaning: lion is OOV, so it becomes -1


### Example 4: `MutableHashTable` (dynamic updates) with OOV

What this does in simple words:
- We start with an empty table.
- Insert tokens and ids now, and insert more later.
- Unknown tokens still return default OOV value (`-1`).

Why useful:
- Helpful when you want to update mapping during preprocessing experiments.

In [15]:
mutable_table = tf.lookup.experimental.MutableHashTable(
    key_dtype=tf.string,
    value_dtype=tf.int64,
    default_value=-1
)

# Insert initial mappings
mutable_table.insert(
    tf.constant(['nlp', 'vision']),
    tf.constant([100, 200], dtype=tf.int64)
)

print('Before update:', mutable_table.lookup(tf.constant(['nlp', 'speech'])).numpy())

# Insert new key later (dynamic behavior)
mutable_table.insert(
    tf.constant(['speech']),
    tf.constant([300], dtype=tf.int64)
)

print('After update :', mutable_table.lookup(tf.constant(['nlp', 'speech', 'robotics'])).numpy())
print('Meaning: robotics is OOV and stays -1')

Before update: [100  -1]
After update : [100 300  -1]
Meaning: robotics is OOV and stays -1


## 12) Quick Revision: All tf.lookup Variants in One Table

| Variant | When to use | OOV style | Static or Dynamic | Pros | Cons |
|---|---|---|---|---|---|
| `tf.lookup.KeyValueTensorInitializer` | Create key-value mapping directly from tensors (used as initializer for tables) | Not applied by itself; OOV behavior comes from table using it | Setup component (not a table) | Very simple and explicit setup in code | Cannot do lookup alone; must be attached to a table |
| `tf.lookup.StaticHashTable` | Fixed dictionary mapping where unknown keys should go to one fallback value | Single default fallback (example: `-1`) | Static | Fast, deterministic, easy for production | All unknowns collapse to one value |
| `tf.lookup.StaticVocabularyTable` | Vocabulary/id mapping where unknown words should be distributed | OOV buckets (`num_oov_buckets > 0`) | Static | Better unknown handling than single fallback, common in NLP | Slightly more setup; bucket ids are hashed, not semantic |
| `tf.lookup.TextFileInitializer` (+ `StaticHashTable`) | Load vocabulary/mapping from file for reproducible pipelines | Usually single default fallback from table (example: `-1`) | Static | Easy to manage large vocab outside code, reproducible | Needs file management and version control |
| `tf.lookup.experimental.MutableHashTable` | Need to insert/update mapping during runtime or experiments | Default fallback for unseen keys (example: `-1`) | Dynamic | Can update table without rebuild | Harder to keep reproducible across runs/deployments |

### One-line Memory Trick
- Need fixed dictionary with one unknown id: use `StaticHashTable`.
- Need vocabulary with OOV buckets: use `StaticVocabularyTable`.
- Need vocab from file: use `TextFileInitializer` + `StaticHashTable`.
- Need runtime updates: use `MutableHashTable`.

## 13) Interview Tips and Enterprise ML App Building Tips

### A) Interview Tips (Tokenizer + Lookup)

1. Start with problem framing
- Say what the business needs first (search relevance, intent detection, multilingual support), then pick tokenizer.

2. Explain tokenizer choice with trade-off language
- Example: "I chose `WordpieceTokenizer` because OOV handling is better than pure whitespace splitting, and we need robust production behavior."

3. Be explicit about OOV strategy
- Mention whether you used single fallback id (`-1`/`[UNK]`) or OOV buckets.
- Interviewers like hearing how unknown tokens are handled in real traffic.

4. Show static vs dynamic design thinking
- Static: stable and reproducible.
- Dynamic: adaptive to new words but needs versioning and governance.

5. Mention evaluation metrics, not only code
- Offline: token coverage, OOV rate, downstream model metrics.
- Online: latency, error rate, conversion/CTR/task success.

### One-line Interview Memory Trick
- "Choice, OOV, Drift, Metrics" -> choose tokenizer, define OOV handling, handle vocabulary drift, measure impact.

---

### B) Enterprise ML App Building Tips

1. Version everything
- Version tokenizer artifact, vocabulary/model file, preprocessing code, and ML model together.
- Keep compatibility matrix: `tokenizer_version` <-> `model_version`.

2. Build a preprocessing contract
- Define input schema, output schema, max length, casing rule, normalization rule, and special tokens.
- Keep same contract in training and serving.

3. Treat OOV and drift as production signals
- Track OOV rate by region/language/use-case.
- Rising OOV rate is usually an early warning for data drift.

4. Keep rollback simple
- If a new tokenizer release hurts quality, rollback to last stable tokenizer+model pair quickly.

5. Add guardrails for multilingual/noisy text
- Use script-aware or subword tokenizers when user text is noisy or multilingual.
- Validate tokenization quality on real samples, not only clean benchmark text.

6. Optimize for latency and cost
- Benchmark tokenizer throughput and memory.
- For high-QPS systems, avoid heavy runtime adaptation unless strongly justified.

7. Test like a product team
- Unit tests: token mapping, special tokens, OOV behavior.
- Integration tests: end-to-end prediction pipeline.
- Shadow/canary rollout before full release.

### One-line Enterprise Memory Trick
- "Version, Contract, Monitor, Rollback" -> version artifacts, enforce preprocessing contract, monitor drift/OOV, keep rollback ready.